In [4]:
import os
import subprocess
import json

def get_video_duration_ffprobe(video_path):
    """Get video duration using ffprobe (more reliable than cv2)"""
    try:
        cmd = [
            'ffprobe', '-v', 'quiet', '-print_format', 'json', 
            '-show_format', video_path
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            data = json.loads(result.stdout)
            return float(data['format']['duration'])
    except (subprocess.TimeoutExpired, json.JSONDecodeError, KeyError, FileNotFoundError):
        pass
    return None

def get_video_duration_cv2(video_path):
    """Fallback method using cv2 if available"""
    try:
        import cv2
        cap = cv2.VideoCapture(video_path)
        if cap.isOpened():
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            duration = frame_count / fps if fps > 0 else None
            cap.release()
            return duration
    except (ImportError, AttributeError):
        pass
    return None

def get_video_duration_mediainfo(video_path):
    """Alternative method using mediainfo if available"""
    try:
        cmd = ['mediainfo', '--Output=JSON', video_path]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            data = json.loads(result.stdout)
            tracks = data.get('media', {}).get('track', [])
            for track in tracks:
                if track.get('@type') == 'Video' and 'Duration' in track:
                    return float(track['Duration'])
    except (subprocess.TimeoutExpired, json.JSONDecodeError, KeyError, FileNotFoundError):
        pass
    return None

## Total Dataset Duration

In [11]:
# Main video processing
video_folder = '/home/nele_pauline_suffo/quantex/quantex_share/RawData'
video_total_seconds = 0
processed_count = 0
failed_count = 0

print(f"Found {len(os.listdir(video_folder))} files in {video_folder}")
print("Processing videos...")

for filename in os.listdir(video_folder):
    if filename.lower().endswith('.mp4'):
        video_path = os.path.join(video_folder, filename)
        
        # Try multiple methods to get duration
        duration = None
        
        # Method 1: ffprobe (most reliable)
        duration = get_video_duration_ffprobe(video_path)
            
        # Method 2: cv2 fallback
        if duration is None:
            duration = get_video_duration_cv2(video_path)
        
        # Method 3: mediainfo fallback
        if duration is None:
            duration = get_video_duration_mediainfo(video_path)
        
        if duration:
            video_total_seconds += duration
            processed_count += 1
        else:
            print("✗ Failed to get duration")
            failed_count += 1

print(f"\nProcessed: {processed_count} videos successfully, {failed_count} failed")

video_hours = int(video_total_seconds // 3600)
video_remaining_seconds = video_total_seconds % 3600
video_minutes = int(video_remaining_seconds // 60)
video_final_seconds = video_remaining_seconds % 60

print(f"Total video duration: {video_hours} hours, {video_minutes} minutes, and {video_final_seconds:.2f} seconds")

Found 356 files in /home/nele_pauline_suffo/quantex/quantex_share/RawData
Processing videos...

Processed: 354 videos successfully, 0 failed
Total video duration: 109 hours, 2 minutes, and 46.67 seconds


## Annotated Subset Duration

In [9]:
import os

# Replace these with your actual video and XML folder paths
video_folder = '/home/nele_pauline_suffo/quantex/quantex_share/RawData/SA_annotation_videos'
json_folder = '/home/nele_pauline_suffo/ProcessedData/childlens_annotations/keeper/v1'

video_total_seconds = 0
processed_count = 0
failed_count = 0

print(f"Found {len(os.listdir(video_folder))} video files in {video_folder}")
print(f"Found {len(os.listdir(json_folder))} JSON files in {json_folder}")
print("Processing videos with matching JSON files...")

# Create a set of base filenames that have a corresponding .xml
xml_basenames = {os.path.splitext(f)[0] for f in os.listdir(json_folder) if f.lower().endswith('.json')}

for filename in os.listdir(video_folder):
    if filename.lower().endswith('.mp4'):
        base_name = os.path.splitext(filename)[0]
        if base_name in xml_basenames:
            video_path = os.path.join(video_folder, filename)

            # Try multiple methods to get duration
            duration = None

            # Method 1: ffprobe (most reliable)
            duration = get_video_duration_ffprobe(video_path)

            # Method 2: cv2 fallback
            if duration is None:
                duration = get_video_duration_cv2(video_path)

            # Method 3: mediainfo fallback
            if duration is None:
                duration = get_video_duration_mediainfo(video_path)

            if duration:
                video_total_seconds += duration
                processed_count += 1
            else:
                print(f"✗ Failed to get duration for {filename}")
                failed_count += 1

print(f"\nProcessed: {processed_count} videos successfully, {failed_count} failed")

video_hours = int(video_total_seconds // 3600)
video_remaining_seconds = video_total_seconds % 3600
video_minutes = int(video_remaining_seconds // 60)
video_final_seconds = video_remaining_seconds % 60

print(f"Total video duration (with matching JSONs): {video_hours} hours, {video_minutes} minutes, and {video_final_seconds:.2f} seconds")

Found 352 video files in /home/nele_pauline_suffo/quantex/quantex_share/RawData/SA_annotation_videos
Found 193 JSON files in /home/nele_pauline_suffo/ProcessedData/childlens_annotations/keeper/v1
Processing videos with matching JSON files...

Processed: 192 videos successfully, 0 failed
Total video duration (with matching JSONs): 54 hours, 44 minutes, and 12.32 seconds


## Activity Class Statistics

In [7]:
import pandas as pd
import os
import json
# Folder containing JSON annotation files
json_folder = '/home/nele_pauline_suffo/ProcessedData/childlens_annotations/keeper/v1'

activity_data = []

for filename in os.listdir(json_folder):
    if filename.endswith('.json'):
        with open(os.path.join(json_folder, filename), 'r') as f:
            data = json.load(f)
            annotations = data.get('annotations', [])
            for ann in annotations:
                event_id = ann.get('eventId')
                duration = ann.get('duration', 0)
                
                # Handle location annotations separately
                if event_id == 'location':
                    location_type = ann.get('fields', {}).get('Type of Location', 'Unknown')
                    category = f"location_{location_type}"
                else:
                    category = event_id
                
                activity_data.append({'activity_class': category, 'duration': duration})

# Create DataFrame
df = pd.DataFrame(activity_data)

# Group by activity class
summary = df.groupby('activity_class').agg(
    num_instances=('activity_class', 'count'),
    total_duration_min=('duration', lambda x: sum(x) / 60)
).reset_index()

print(summary)

                  activity_class  num_instances  total_duration_min
0                  child_talking          12093          839.036562
1                crafting_things             28          106.997550
2                        dancing             21            8.751122
3                        drawing             83          366.300287
4                        exclude              3            1.534824
5   listening_to_music/audiobook             96          322.875620
6                      location_              1            0.650000
7              location_Bathroom              2            2.247294
8               location_Hallway             18           42.469450
9            location_Livingroom            112         1590.987691
10                 location_None              4           81.379632
11                location_Other             49          559.877609
12             location_Playroom             71          998.733441
13              location_Unknown              1 

## Exclude Category

In [12]:
import os
import json

# Iterate through all .json files in the directory
file_dir = '/home/nele_pauline_suffo/ProcessedData/childlens_annotations/keeper/v1'
exclude_found = False

for filename in os.listdir(file_dir):
    filepath = os.path.join(file_dir, filename)
    if filepath.endswith('.json'):
        with open(filepath, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except Exception as e:
                print(f"Error loading {filepath}: {e}")
                continue

            # Check if annotations exist and search for "Exclude" eventId
            if 'annotations' in data and isinstance(data['annotations'], list):
                for annotation in data['annotations']:
                    if isinstance(annotation, dict) and annotation.get('eventId') == 'exclude':
                        print(f'File: {filepath}')
                        print(f'Video: {data.get("video_name", "Unknown")}')
                        print(f'Exclude annotation: {annotation}\n')
                        exclude_found = True

if not exclude_found:
    print("No 'Exclude' annotations found in any files.")

File: /home/nele_pauline_suffo/ProcessedData/childlens_annotations/keeper/v1/572947.json
Video: 572947.MP4
Exclude annotation: {'categoryId': 'human_actions', 'eventId': 'exclude', 'fields': {'Type of Scenes': 'Nudity'}, 'time': 131.091753, 'type': 'complete', 'videoName': '572947.MP4', 'startTime': 131.091753, 'endTime': 221.000063, 'duration': 89.90831}

File: /home/nele_pauline_suffo/ProcessedData/childlens_annotations/keeper/v1/364368.json
Video: 364368.MP4
Exclude annotation: {'time': 34.05841, 'categoryId': 'human_actions', 'eventId': 'exclude', 'fields': {'Alone?': '', '1st Person Age Group': '', '1st Person Gender': '', '2nd Person Age Group': '', '2nd Person Gender': '', '3rd Person Age Group': '', '3rd Person Gender': '', '4th Person Age Group': '', '4th Person Gender': '', '5th Person Age Group': '', '5th Person Gender': '', '6th Person Age Group': '', '6th Person Gender': ''}, 'type': 'complete', 'videoName': '364368.MP4', 'startTime': 34.05841, 'endTime': 34.875229, 'durat